# 🚀 V6.1 微服务化系统集成示例

本 Notebook 演示 V6.1 微服务化架构的完整集成流程：
- ✅ 11个独立微服务初始化
- ✅ 通信层（ServiceRegistry/MessageBus/APIGateway）
- ✅ 基础服务层（Cache/Config/IndexMapping）
- ✅ 业务服务层（市场状态/风险评估/资产配置等）
- ✅ 可视化服务（18大图表完整生成）
- ✅ data_context 准备与验证
- ✅ HTML报告导出

In [1]:
# ==================== 1. 导入必要模块 ====================
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

# 添加项目根目录到PATH
project_root = Path('../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# ==================== 方案1：最简用法（推荐） ====================
from utils.config_utils import extract_config_dict
from infrastructure.base_services.config_service import ConfigService
# ✅ 无需指定路径！自动检测项目根目录
config = ConfigService()  # 自动加载 config/system_config_v6.yaml
print(f"✅ 配置加载成功 | 项目根目录: {config.config_path.parent.parent}")
# 导入基础服务
from infrastructure.base_services.config_service import ConfigService
from infrastructure.base_services.cache_service import CacheService
from infrastructure.base_services.index_mapping_service import IndexMappingService
from infrastructure.data_service.data_loading_service import DataLoadingService

# 导入通信层
from infrastructure.communication_layer.service_registry import ServiceRegistry
from infrastructure.communication_layer.message_bus import MessageBus
from infrastructure.communication_layer.api_gateway import APIGateway

# 导入核心业务服务（7个）
from services.core_services.market_state_service import MarketStateService
from services.core_services.risk_assessment_service import RiskAssessmentService
from services.core_services.allocation_service import AllocationService
from services.core_services.sentiment_analysis_service import SentimentAnalysisService
from services.core_services.commodity_engine_service import CommodityEngineService
from services.core_services.macro_analysis_service import MacroAnalysisService
from services.core_services.option_pcr_service import OptionPCRService

# 导入补充分析服务（3个新增）
from services.supplementary_services.cross_market_service import CrossMarketService
from services.supplementary_services.industry_rotation_service import IndustryRotationService
from services.supplementary_services.futures_analysis_service import FuturesAnalysisService

# 导入可视化服务（18大图表）
from services.visualization_service.visualization_service import VisualizationService

# 导入工具函数
from utils.data_context_preparation import prepare_visualization_data_context , validate_data_context
from utils.validation_utils import validate_dataframe, validate_numeric_range
from utils.type_conversion_utils import ensure_python_float, ensure_python_int

from services.threshold_service.threshold_service import ThresholdService  # ✅ V6.1新增

# 导入主系统（修复版）
from main_system.market_state_system_v6 import MarketStateSystem

print("✅ 所有模块导入成功")

✅ 配置加载成功 | 项目根目录: /home/ts/app/AiStock/market_state_system_v6
✅ 所有模块导入成功


In [2]:
# ==================== 2. 初始化基础服务层 ====================
print("=" * 80)
print("🔧 初始化基础服务层...")
print("=" * 80)

# 配置服务（修复版：递归转换YAML→dataclass）
config_service = ConfigService('system_config_v6.yaml')
print(f"✅ ConfigService: 配置加载成功 | adaptive_config类型: {type(config_service.config.get('adaptive_config'))}")

# 缓存服务（LRU+TTL）
# 初始化缓存（可选：使用系统默认配置）
cache_service = CacheService(
    max_size=config.config.get('cache', {}).get('max_size', 1000),
    ttl=config.config.get('cache', {}).get('ttl', 3600)
)
print(f"✅ CacheService: 容量={cache_service.max_size}, TTL={cache_service.ttl}s")

# 指数映射服务（代码→名称）
index_mapping = IndexMappingService('index_name_mapping.yaml')
print(f"✅ IndexMappingService: 指数={len(index_mapping.get_csi_indices())} | 期货={len(index_mapping.get_futures())}")

# 数据加载服务（集成缓存）
data_service = DataLoadingService(config_service, cache_service,enable_cache=False)  # 可选：启用缓存)
print(f"✅ DataLoadingService: 初始化完成")

print("\n✅ 基础服务层初始化完成")

# 阈值动态化服务（V6.1新增）
threshold_service = ThresholdService(config_service)

🔧 初始化基础服务层...
✅ ConfigService: 配置加载成功 | adaptive_config类型: <class 'dict'>
✅ CacheService: 容量=1000, TTL=3600s
✅ IndexMappingService: 指数=215 | 期货=80
✅ DataLoadingService: 初始化完成

✅ 基础服务层初始化完成


In [3]:
# ==================== 3. 初始化通信层 ====================
print("\n" + "=" * 80)
print("📡 初始化通信层...")
print("=" * 80)

# 服务注册中心（健康检查+依赖追踪）
service_registry = ServiceRegistry(heartbeat_timeout=30)
print(f"✅ ServiceRegistry: 心跳超时={service_registry.heartbeat_timeout}s")

# 消息总线（发布/订阅）
message_bus = MessageBus(max_queue_size=1000)
message_bus.start()
print(f"✅ MessageBus: 队列容量={message_bus.message_queue.maxsize} | 线程启动")

# API网关（限流+熔断）
api_gateway = APIGateway(
    service_registry=service_registry,
    message_bus=message_bus,
    enable_rate_limit=True,
    enable_circuit_breaker=True
)
print(f"✅ APIGateway: 限流={api_gateway.enable_rate_limit} | 熔断={api_gateway.enable_circuit_breaker}")

print("\n✅ 通信层初始化完成")


📡 初始化通信层...
✅ ServiceRegistry: 心跳超时=30s
✅ MessageBus: 队列容量=1000 | 线程启动
✅ APIGateway: 限流=True | 熔断=True

✅ 通信层初始化完成


In [4]:
# ==================== 4. 初始化业务服务层（11个独立微服务）====================
print("\n" + "=" * 80)
print("⚙️ 初始化业务服务层（11个独立微服务）...")
print("=" * 80)

# 核心业务服务（7个）
market_state_service = MarketStateService(data_service, config_service, threshold_service)
risk_service = RiskAssessmentService(data_service, config_service, threshold_service)
allocation_service = AllocationService(config_service, threshold_service)
sentiment_service = SentimentAnalysisService(data_service, config_service, threshold_service)
commodity_service = CommodityEngineService(data_service, config_service, threshold_service)
macro_service = MacroAnalysisService(data_service, config_service, threshold_service)
pcr_service = OptionPCRService(data_service, config_service, threshold_service)

# 补充分析服务（3个新增）
cross_market_service = CrossMarketService(data_service, config_service, threshold_service)
rotation_service = IndustryRotationService(data_service, config_service, threshold_service)
futures_service = FuturesAnalysisService(data_service, config_service, threshold_service)
mapper_service = IndexMappingService('index_name_mapping.yaml')  # ✅ 独立指数映射服务
# 可视化服务（18大图表完整实现）
viz_service = VisualizationService({
    'chinese_font': "Microsoft YaHei, SimHei, sans-serif",
    'export_path': './reports/v6_visualization/'
}, config_service=config_service, index_mapper=mapper_service)  # ✅ 注入独立指数映射服务

print(f"✅ MarketStateService: 市场状态判定（九宫格定位）")
print(f"✅ RiskAssessmentService: 微盘熔断+prepare_high_risk_data()")
print(f"✅ AllocationService: 九大战略方向动态配置")
print(f"✅ SentimentAnalysisService: calculate_fund_flow_heatmap()完整修复")
print(f"✅ CommodityEngineService: 商品信号+期限结构")
print(f"✅ MacroAnalysisService: 五维宏观评分")
print(f"✅ OptionPCRService: 动态合约识别+综合PCR")
print(f"✅ CrossMarketService: A股/港股/美股/汇率/美债联动")
print(f"✅ IndustryRotationService: 行业轮动矩阵")
print(f"✅ FuturesAnalysisService: calculate_index_futures_basis()新增（股指基差）")
print(f"✅ VisualizationService: 18大图表完整实现（Plotly类型修复）")

print("\n✅ 业务服务层初始化完成（11个独立微服务，无循环依赖）")


⚙️ 初始化业务服务层（11个独立微服务）...
✅ MarketStateService: 市场状态判定（九宫格定位）
✅ RiskAssessmentService: 微盘熔断+prepare_high_risk_data()
✅ AllocationService: 九大战略方向动态配置
✅ SentimentAnalysisService: calculate_fund_flow_heatmap()完整修复
✅ CommodityEngineService: 商品信号+期限结构
✅ MacroAnalysisService: 五维宏观评分
✅ OptionPCRService: 动态合约识别+综合PCR
✅ CrossMarketService: A股/港股/美股/汇率/美债联动
✅ IndustryRotationService: 行业轮动矩阵
✅ FuturesAnalysisService: calculate_index_futures_basis()新增（股指基差）
✅ VisualizationService: 18大图表完整实现（Plotly类型修复）

✅ 业务服务层初始化完成（11个独立微服务，无循环依赖）


In [ ]:
# ==================== 5. 初始化主系统（修复版）====================
print("\n" + "=" * 80)
print("🚀 初始化主系统（MarketStateSystemV6_1）...")
print("=" * 80)

system = MarketStateSystem('system_config_v6.yaml')

print(f"✅ 主系统初始化成功 | 基准日: {system.config.config.get('base_date', 'today')}")
# print(f"✅ 业务数据归属: system.benchmark_data（非DataManager）")
print(f"✅ 服务解耦: 所有服务仅通过参数传递数据")

In [ ]:
# # ==================== 6. 运行系统（完整分析流程）====================
# print("\n" + "=" * 80)
# print("📊 运行V6.0系统（完整市场分析）...")
# print("=" * 80)

# result = system.run()

# print(f"\n🎯 市场状态: {result['market_state']}")
# print(f"📊 估值安全边际: {result['valuation_score']:.1f}/100")
# print(f"📈 趋势动能强度: {result['trend_score']:.1f}/100")

# if result.get('micro_liquidity'):
#     print(f"🔥 微盘熔断阶段: {result['micro_liquidity']['stage']} (持续{result['micro_liquidity']['days_in_stage']}日)")

# print(f"\n💼 九大战略方向配置（前5大）:")
# df_no_cash = result['allocation'][result['allocation']['战略方向'] != '现金'].copy()
# top5 = df_no_cash.nlargest(5, '动态权重')
# for _, row in top5.iterrows():
#     print(f"  • {row['战略方向']:8s} | {row['配置建议']:6s} | {row['核心指数']}")

# print(f"\n⚠️ 风险预警（前3条）:")
# for i, alert in enumerate(result['risk_alerts'][:3], 1):
#     print(f"  {i}. {alert}")

In [5]:
# ==================== 7. 准备data_context（18大图表所需全部数据）====================
print("\n" + "=" * 80)
print("📦 准备data_context（18大图表所需全部数据）...")
print("=" * 80)

# 使用工具函数准备完整data_context
data_context = prepare_visualization_data_context(
    market_state_service=market_state_service,
    risk_service=risk_service,
    allocation_service=allocation_service,
    sentiment_service=sentiment_service,
    commodity_service=commodity_service,
    macro_service=macro_service,
    pcr_service=pcr_service,
    cross_market_service=cross_market_service,
    rotation_service=rotation_service,
    futures_service=futures_service,
    data_service=data_service,
    config_service=config_service,
    # benchmark_data=None,          # ✅ 新增参数
    # micro_liquidity=None          # ✅ 新增参数  
)

# 验证完整性
is_valid, missing_keys = validate_data_context(data_context)
if is_valid:
    print("✅ data_context验证通过 | 所有必需数据已准备")
else:
    print(f"⚠️ data_context缺失: {missing_keys}")

# 统计各数据项
print("\n📊 data_context数据统计:")
for key in [
    'market_state', 'val_score', 'trend_score',
    'benchmark_data', 'micro_data', 'allocation_df',
    'pcr_data', 'basis_data', 'term_data',
    'flow_data', 'sentiment_data', 'market_data',
    'industry_data', 'risk_metrics', 'risk_data',
    'commodity_signals', 'macro_history',
    'pe_data', 'bond_yield'
]:
    value = data_context.get(key)
    if isinstance(value, dict):
        print(f"   • {key:20s}: {len(value)} 项")
    elif isinstance(value, list):
        print(f"   • {key:20s}: {len(value)} 项")
    elif isinstance(value, pd.DataFrame):
        print(f"   • {key:20s}: {len(value)} 行")
    else:
        print(f"   • {key:20s}: {value}")


📦 准备data_context（18大图表所需全部数据）...


❌ 数据库加载衍生品失败 HO8R03H1: Execution failed on sql '
                SELECT d
❌ 数据库加载衍生品失败 HO8R03JT: Execution failed on sql '
                SELECT d
⚠️ 未成功加载任何510300的平值合约K线 | 尝试合约: 2个 | 原因: TDX接口可能不可用或合约代码无效
⚠️ 510300 期权数据加载失败（市场代码: 8），触发降级策略
❌ 数据库加载衍生品失败 HO8R03H1: Execution failed on sql '
                SELECT d
❌ 数据库加载衍生品失败 HO8R03JT: Execution failed on sql '
                SELECT d
⚠️ 未成功加载任何510500的平值合约K线 | 尝试合约: 2个 | 原因: TDX接口可能不可用或合约代码无效
⚠️ 510500 期权数据加载失败（市场代码: 8），触发降级策略
❌ 数据库加载衍生品失败 HO8R03H1: Execution failed on sql '
                SELECT d
❌ 数据库加载衍生品失败 HO8R03JT: Execution failed on sql '
                SELECT d
⚠️ 未成功加载任何588000的平值合约K线 | 尝试合约: 2个 | 原因: TDX接口可能不可用或合约代码无效
⚠️ 588000 期权数据加载失败（市场代码: 8），触发降级策略
❌ 数据库加载衍生品失败 HO8R03H1: Execution failed on sql '
                SELECT d
❌ 数据库加载衍生品失败 HO8R03JT: Execution failed on sql '
                SELECT d
⚠️ 未成功加载任何159915的平值合约K线 | 尝试合约: 2个 | 原因: TDX接口可能不可用或合约代码无效
⚠️ 159915 期权数据加载失败（市场代码: 9），触发降级策略
❌ 数据库加载衍生品失败 CU2603: Execution f

⚠️ data_context缺失: ['term_data']

📊 data_context数据统计:
   • market_state        : 防御进攻区
   • val_score           : 12.9471218206158
   • trend_score         : 48.02207637869452
   • benchmark_data      : 4 项
   • micro_data          : 3 项
   • allocation_df       : 10 行
   • pcr_data            : 6 项
   • basis_data          : 4 项
   • term_data           : 0 项
   • flow_data           : 2 项
   • sentiment_data      : 4 项
   • market_data         : 6 项
   • industry_data       : 2 项
   • risk_metrics        : 4 项
   • risk_data           : 5 项
   • commodity_signals   : 9 项
   • macro_history       : 3 项
   • pe_data             : 1489 行
   • bond_yield          : 2.5


In [6]:
data_service.load_derivative_data('HO8R03H1', market_code=33, days=5)

❌ 数据库加载衍生品失败 HO8R03H1: Execution failed on sql '
                SELECT d


""


In [ ]:
data_context.get('benchmark_data')  # 示例访问基差数据

In [ ]:
config_service.get('option_underlying_mapping', {})

In [11]:
data_service.load_derivative_data('HO8R03H1',7)

,open,high,low,close,open_interest,volume,settlement,year,month,day,hour,minute,datetime,turnover
0,708.400024,708.400024,708.400024,708.400024,29.0,0.0,712.200012,2025.0,11.0,28.0,15.0,0.0,2025-11-28 15:00:00,4.063766e-44
1,722.000000,728.400024,720.200012,728.400024,30.0,0.0,729.000000,2025.0,12.0,1.0,15.0,0.0,2025-12-01 15:00:00,4.203895e-44
2,726.200012,726.200012,723.799988,723.799988,29.0,0.0,718.799988,2025.0,12.0,2.0,15.0,0.0,2025-12-02 15:00:00,4.063766e-44
3,709.200012,709.200012,707.799988,707.799988,27.0,0.0,705.000000,2025.0,12.0,3.0,15.0,0.0,2025-12-03 15:00:00,3.783506e-44
4,707.200012,707.200012,706.400024,706.400024,27.0,0.0,712.200012,2025.0,12.0,4.0,15.0,0.0,2025-12-04 15:00:00,3.783506e-44
5,743.400024,743.400024,740.000000,740.000000,27.0,0.0,740.799988,2025.0,12.0,5.0,15.0,0.0,2025-12-05 15:00:00,3.783506e-44
6,756.000000,759.400024,754.799988,759.400024,30.0,0.0,757.400024,2025.0,12.0,8.0,15.0,0.0,2025-12-08 15:00:00,4.203895e-44
7,735.599976,736.400024,735.599976,736.400024,33.0,0.0,734.400024,2025.0,12.0,9.0,15.0,0.0,2025-12-09 15:00:00,4.624285e-44
8,723.400024,723.400024,713.599976,721.799988,35.0,0.0,721.799988,2025.0,12.0,10.0,15.0,0.0,2025-12-10 15:00:00,4.904545e-44
9,711.400024,711.400024,711.400024,711.400024,36.0,0.0,711.400024,2025.0,12.0,11.0,15.0,0.0,2025-12-11 15:00:00,5.044674e-44


In [ ]:
# ==================== 8. 生成18大可视化图表 ====================
print("\n" + "=" * 80)
print("🎨 生成18大可视化图表...")
print("=" * 80)

charts = viz_service.generate_all_charts(data_context)

valid_charts = {k: v for k, v in charts.items() if v is not None}
print(f"\n✅ 成功生成 {len(valid_charts)}/{len(charts)} 个图表")

# 显示生成的图表列表
print("\n📊 生成的图表列表:")
for i, (name, chart) in enumerate(valid_charts.items(), 1):
    status = "✅" if chart else "❌"
    print(f"  {status} {i:2d}. {name}")

In [ ]:
data_context.keys()

In [ ]:
data_context.get('pcr_data').get('components')

In [ ]:
data_context.get('pcr_data')

In [ ]:
viz_service._generate_valuation_diagnostic_chart(data_context.get('valuation_score'))

In [ ]:
# # ==================== 9. 在Jupyter中显示前5个图表 ====================
# print("\n" + "=" * 80)
# print("🖥️ 在Jupyter中显示前5个图表...")
# print("=" * 80)

# try:
#     from IPython.display import display, Markdown, HTML
    
#     # 显示头部
#     display(HTML(f"""
#     <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
#                 color: white; padding: 25px; border-radius: 15px; margin-bottom: 30px;">
#         <h1 style="text-align: center; margin: 0; font-size: 32px;">
#             📈 A股市场状态量化系统 V6.0 - 可视化报告
#         </h1>
#         <p style="text-align: center; margin: 10px 0 0 0; font-size: 18px;">
#             微服务化架构 | 18大交互式图表 | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
#         </p>
#     </div>
#     """))
    
#     # 显示市场状态摘要
#     display(Markdown("### 🎯 市场状态摘要"))
#     display(Markdown(f"**当前市场状态**: {data_context['market_state']}"))
#     display(Markdown(f"**估值安全边际**: {data_context['val_score']:.1f}/100"))
#     display(Markdown(f"**趋势动能强度**: {data_context['trend_score']:.1f}/100"))
    
#     if data_context.get('micro_data', {}).get('liquidity_status'):
#         stage = data_context['micro_data']['liquidity_status']['stage']
#         days = data_context['micro_data']['liquidity_status']['days_in_stage']
#         display(Markdown(f"**微盘熔断阶段**: {stage} (持续{days}日)"))
    
#     # 显示前5个图表
#     display(Markdown("\n### 📊 可视化图表（前5个）"))
#     for i, (name, chart) in enumerate(list(valid_charts.items())[:5], 1):
#         display(Markdown(f"#### {i}. {name}"))
#         if chart:
#             display(chart)
#         else:
#             display(Markdown(f"⚠️ {name} 图表生成失败"))
    
#     print("✅ Jupyter图表显示完成")
    
# except Exception as e:
#     print(f"⚠️ Jupyter显示失败（不影响报告导出）: {str(e)[:50]}")

In [ ]:
# ==================== 10. 导出HTML可视化报告 ====================
print("\n" + "=" * 80)
print("📤 导出HTML可视化报告...")
print("=" * 80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"./reports/visualization_report_v6_{timestamp}.html"

report_path = viz_service.export_charts_to_html(
    charts=charts,
    output_path=output_path,
    title="A股市场状态量化系统 V6.0 - 完整可视化报告"
)

if report_path:
    print(f"\n✅ 可视化报告已导出至: {report_path}")
    print(f"🌐 在浏览器中打开: file://{os.path.abspath(report_path)}")
    
    # 可选：自动打开浏览器（取消注释）
    # import webbrowser
    # webbrowser.open('file://' + os.path.abspath(report_path))
else:
    print("❌ 报告导出失败")

In [ ]:
# ==================== 11. 系统清理 ====================
print("\n" + "=" * 80)
print("🧹 系统清理...")
print("=" * 80)

# 停止消息总线
message_bus.stop()
print("✅ MessageBus: 已停止")

# 清理缓存（可选）
# cache_service.clear()
# print("✅ CacheService: 缓存已清空")

print("\n" + "=" * 80)
print("✅ V6.0微服务化系统集成示例运行完成")
print("=" * 80)
print("\n💡 下一步建议:")
print("  1. 在浏览器中打开HTML报告")
print("  2. 修改config/system_config_v6.yaml调整阈值")
print("  3. 运行scripts/health_check.sh验证系统完整性")
print("  4. 查看docs/user_guide/quick_start.md获取更多指南")

In [ ]:
data_context.get('pcr_data')

In [ ]:
data_context.get('pcr_data').get('components', {}).get('510300')

In [ ]:
viz_service._generate_option_pcr_chart(data_context.get('pcr_data', {}))

##### 单元测试

In [7]:
# ==================== 1. 导入必要模块 ====================
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

# 添加项目根目录到PATH
project_root = Path('../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [8]:
from infrastructure.base_services.config_service import ConfigService
from infrastructure.data_service.data_loading_service import DataLoadingService
from services.core_services.option_pcr_service import OptionPCRService


config = ConfigService('system_config_v6.yaml')
data_service = DataLoadingService(config)
pcr_service = OptionPCRService(data_service, config)

In [ ]:
data_context.get('pcr_data')

In [ ]:
config.get('option_underlying_mapping', {})

In [ ]:
config.get('option_underlying_mapping', {}).get('510300')

In [ ]:
data_service.load_derivative_data('510300', 33).tail(1)

In [ ]:
# 测试1：基础容忍度获取（应为0.05）
tolerance1 = pcr_service._get_dynamic_tolerance('510300', market_code=8, vol_percentile=50.0)
print(f"✅ 基础容忍度 (50%波动): {tolerance1:.3f} | 期望: 0.050")
assert abs(tolerance1 - 0.05) < 0.01, "基础容忍度错误"

# 测试2：高波动调整（应为0.08×1.0=0.08）
tolerance2 = pcr_service._get_dynamic_tolerance('510300', market_code=8, vol_percentile=85.0)
print(f"✅ 高波动容忍度 (85%波动): {tolerance2:.3f} | 期望: 0.080")
assert abs(tolerance2 - 0.08) < 0.01, "高波动调整错误"

# 测试3：中金所微调（应为0.05×1.2=0.06）
tolerance3 = pcr_service._get_dynamic_tolerance('IO', market_code=7, vol_percentile=50.0)
print(f"✅ 中金所容忍度 (50%波动): {tolerance3:.3f} | 期望: 0.060")
assert abs(tolerance3 - 0.06) < 0.01, "中金所微调错误"

# 测试4：类型安全（必须为 Python 原生 float）
print(f"✅ 类型验证: {type(tolerance1).__name__} | 是否 numpy: {isinstance(tolerance1, np.floating)}")
assert isinstance(tolerance1, float) and not isinstance(tolerance1, np.floating), "类型错误"

print("\n🎉 _get_dynamic_tolerance 修复验证通过！")